##Google ColabからGoogleドライブのファイルにアクセスできるようにするためのコード

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##必要なライブラリのインポート

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import MeanIoU
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import json
from PIL import Image
import cv2

##学習済みモデルの読み込み
保存しているモデル（~.h5）のパスを設定
今回は、 RARP_BND という名前のフォルダに保存している ZEUS_final.h5 を使用する設定
学習済みモデルは以下よりダウンロード可能
https://doi.org/10.5281/zenodo.17010634

In [ ]:
model_path = '/content/drive/MyDrive/RARP_BND/ZEUS_final.h5'
model = load_model(model_path)

##セグメンテーションのカラーマップ（色分け）の設定
最終的なモデルのクラス分け（判断する物体の種類数）に合わせて、カラーの数を変更する。
ここに記載しているモデルであるZEUS_final.h5のクラスは「9クラス＋1（background）」の合計10クラスとなり、それぞれに対応する10色を設定している。
*別のモデルを使用する際は、そのモデルの出力クラスに合わせてここを書き換える。

In [ ]:
colors = {
    0: [0, 0, 0],       # background - 黒
    1: [255, 0, 0],     # bladder - 青
    2: [0, 255, 0],     # border - 緑
    3: [0, 255, 255],   # catheter - 黄色
    4: [255, 255, 255], # gauze - 白
    5: [128, 128, 128], # instrument - グレー
    6: [0, 0, 255],     # mucosa - 赤
    7: [255, 0, 255],   # prostate - ピンク
    8: [128, 128, 128], # suction - グレー
    9: [128, 0, 128]    # urethra - 紫
}

##動画を読み込み、セグメンテーション（色分け）をオーバーレイ（重ね合わせ）した新しい動画を出力するメインのコード
今回は、RARP_BNDフォルダに保存した RARP_BND_video.mp4 を読み込み、処理結果を RARP_BND_video_overlay.mp4 として同フォルダに保存する。

In [ ]:
# 入力動画の読み込み
input_video_path = '/content/drive/MyDrive/RARP_BND/RARP_BND_video.mp4'
cap = cv2.VideoCapture(input_video_path)

# 動画の基本情報を取得
fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_video_path = '/content/drive/MyDrive/RARP_BND/RARP_BND_video_overlay.mp4'
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# フレーム処理
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # フレームの前処理
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image, (256, 256))
    image_array = np.expand_dims(image_resized / 255.0, axis=0)

    # 予測を実行
    prediction = model.predict(image_array)
    predicted_mask = np.argmax(prediction[0], axis=-1)

    # カラーマップの適用
    colormap = np.zeros((predicted_mask.shape[0], predicted_mask.shape[1], 3), dtype=np.uint8)
    for label, color in colors.items():
        colormap[predicted_mask == label] = color

    # 元のサイズにリサイズ
    colormap_resized = cv2.resize(colormap, (frame_width, frame_height))

    # 元のフレームとマスクを重ねる
    overlay = cv2.addWeighted(frame, 0.6, colormap_resized, 0.4, 0)

    # 結果を出力ファイルに書き込み
    out.write(overlay)

cap.release()
out.release()
print("動画の処理が完了しました。")